# 5 Gate, highway, or route: one idea from three papers

The gate/highway/MoE family composes with any `(batch, hidden)` tensor — images, text, tabular data, anything. It sits here because
this lesson already has a window and a baseline table to beat, the same reason
`1_representation_and_head/04_search` put its search where a training run takes seconds: the
place makes the idea cheap to test, not because the idea belongs to one kind of data.


## 5.1 One idea, three resolutions

| component | mechanism | where it comes from |
|---|---|---|
| `Gate` | `sigmoid(Wx) * x` -- a learned per-dimension scale | the gate inside every LSTM cell (Hochreiter & Schmidhuber, 1997) and the reset/update gates of the GRU (Cho et al., 2014, *Learning Phrase Representations using RNN Encoder-Decoder for Statistical Machine Translation*) |
| `Highway` | `g * T(x) + (1 - g) * x` -- a learned interpolation between a transform and the identity | Highway Networks (Srivastava, Greff & Schmidhuber, 2015), built explicitly to let very deep plain networks train at all -- the direct ancestor of the ResNet residual connection, which is a Highway with the gate fixed at 0.5 |
| `MoE` | `sum_i softmax(Wx)_i * expert_i(x)` -- N gates routing between N experts | Jacobs & Jordan, 1991, *Adaptive Mixtures of Local Experts* -- the **dense** version trained in §5.3 below: all N experts run, every time. Modern **sparse** MoE (Shazeer et al., 2017, *Outrageously Large Neural Networks*; the Switch Transformer, Fedus et al., 2021) routes each token to only `k` of thousands of experts -- capacity scales with N, compute scales with k. §5.3 trains that version (`vectormesh`’s `SparseMoE`) next to the dense one, and §5.5 times it, because skipping the unchosen experts is what makes production MoE fast at scale |

Read top to bottom, each row is the row above with more capacity to choose from: one
gate choosing between a transform and nothing (`Gate`), one gate choosing between a
transform and the identity (`Highway`), N gates choosing between N transforms (`MoE`).

This lesson's electricity-demand series has real daily and weekly structure -- exactly
the kind of thing a router could plausibly use. Predict before you look at the table:
does giving the mechanism something real to route on change the verdict?


In [ ]:
from dataclasses import dataclass
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from loguru import logger
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from vectormesh.components import Gate, Highway, MoE, NeuralNet, SparseMoE
from vectormesh.components.metrics import MAE
from vectormesh.training import Trainer, TrainerSettings

from australian_electricity import load_ausalia
from windowing import window_index
from notebooktester import param

# the Trainer logs every epoch at INFO; keep warnings only
logger.remove()
logger.add(sys.stderr, level="WARNING")

DAY = 48  # half-hourly steps per day
# four days of history, one day of horizon: the window 6_uncertainty/01_probabilistic
# uses, so results sit next to that notebook's
W, H = 4 * DAY, DAY
HIDDEN = 64
STRIDE = 24
SEEDS = (0, 1, 2, 3, 4)
# early stopping: training ends after PATIENCE epochs without a lower val loss, and
# the weights of the best epoch are kept. Val MAE moves up and down from one epoch to
# the next by more than the gaps between models (see the curve in 5.2); a patience of
# 10 epochs rides out that bouncing instead of stopping at the first bump.
PATIENCE = 10
MAX_EPOCHS = param(200, test=3)  # a ceiling; early stopping ends every run before it

ausalia = load_ausalia()
mu = ausalia.values[ausalia.splits["train"]].mean()
sigma = ausalia.values[ausalia.splits["train"]].std()


def windows(split: str, stride: int):
    values = (ausalia.values[ausalia.splits[split]] - mu) / sigma
    win = values[window_index(len(values), W + H, stride)]
    return win[:, :W], win[:, W:]


xtr, ytr = windows("train", STRIDE)
xva, yva = windows("val", STRIDE * 2)
print(f"train {tuple(xtr.shape)}   val {tuple(xva.shape)}")

# one batch covering the whole val split -- every Trainer below shares this loader
validloader = DataLoader(TensorDataset(xva, yva), batch_size=len(xva))


def mae(y: torch.Tensor, yhat: torch.Tensor) -> float:
    return float((y - yhat).abs().mean())


def make_model(
    mechanism: str, hidden: int = HIDDEN, n_experts: int = 4, k: int = 1
) -> nn.Module:
    if mechanism == "plain":
        return NeuralNet(hidden_size=[W, hidden], out_size=H)
    if mechanism == "deeper":
        # capacity-matched control: one more Linear -> GELU than "plain", no gate at all.
        # Gate's own weight matrix costs exactly what this second layer costs at this width.
        return NeuralNet(hidden_size=[W, hidden, hidden], out_size=H)
    inp, act, out = nn.Linear(W, hidden), nn.GELU(), nn.Linear(hidden, H)
    if mechanism == "gate":
        mid = Gate(hidden_size=hidden)
    elif mechanism == "highway":
        mid = Highway(transform=NeuralNet(hidden, hidden), hidden_size=hidden)
    elif mechanism == "moe":
        mid = MoE(
            experts=[nn.Linear(hidden, hidden) for _ in range(n_experts)],
            hidden_size=hidden,
            out_size=hidden,
        )
    elif mechanism == "sparse_moe":
        mid = SparseMoE(
            experts=[nn.Linear(hidden, hidden) for _ in range(n_experts)],
            hidden_size=hidden,
            out_size=hidden,
            k=k,
        )
    else:
        raise ValueError(mechanism)
    return nn.Sequential(inp, act, mid, out)


@dataclass
class Run:
    """One training run with early stopping: the val MAE of the best epoch, that epoch,
    the last epoch that ran, and the model holding the best epoch's weights."""

    mae: float
    best_epoch: int
    stopped_epoch: int
    model: nn.Module


def run(
    mechanism: str,
    seed: int,
    max_epochs: int,
    lr: float = 1e-3,
    batch_size: int = 64,
    **model_kwargs,
) -> Run:
    torch.manual_seed(seed)
    model = make_model(mechanism, **model_kwargs)
    trainloader = DataLoader(
        TensorDataset(xtr, ytr), batch_size=batch_size, shuffle=True
    )
    settings = TrainerSettings(
        epochs=max_epochs,
        metrics=[MAE()],
        logdir=Path("modellogs").resolve(),
        train_steps=len(trainloader),
        valid_steps=len(validloader),
        optimizer_kwargs={"lr": lr},
        earlystop_kwargs={"patience": PATIENCE, "save": False, "verbose": False},
    )
    trainer = Trainer(
        model=model,
        settings=settings,
        loss_fn=nn.MSELoss(),
        optimizer=torch.optim.Adam,
        traindataloader=trainloader,
        validdataloader=validloader,
        progress=False,
    )
    # the result and trainer.model describe the epoch with the lowest val loss.
    # TrainResult counts epochs from 0; this notebook counts from 1, as the curve does.
    result = trainer.loop()
    return Run(
        mae=result.metric_dict["MAE"],
        best_epoch=result.epoch + 1,
        stopped_epoch=result.stopped_epoch + 1,
        model=trainer.model,
    )


def mae_curve(
    mechanism: str,
    epochs: int,
    seed: int = 0,
    lr: float = 1e-3,
    batch_size: int = 64,
    **model_kwargs,
) -> tuple[list[float], list[float], list[float]]:
    """The same training as `run`, without early stopping, for a fixed number of epochs.
    Returns train MAE, val MAE and val loss (MSE) after every epoch: a Reporter reads val
    MAE and val loss from the Trainer, and does one extra forward pass over train for
    the train-side curve."""
    torch.manual_seed(seed)
    model = make_model(mechanism, **model_kwargs)
    trainloader = DataLoader(
        TensorDataset(xtr, ytr), batch_size=batch_size, shuffle=True
    )
    train_curve: list[float] = []
    val_curve: list[float] = []
    val_loss: list[float] = []

    def track(epoch, train_loss, test_loss, metric_dict):
        val_curve.append(metric_dict["MAE"])
        val_loss.append(test_loss)
        with torch.no_grad():
            train_curve.append(mae(ytr, model(xtr)))

    settings = TrainerSettings(
        epochs=epochs,
        metrics=[MAE()],
        logdir=Path("modellogs").resolve(),
        train_steps=len(trainloader),
        valid_steps=len(validloader),
        optimizer_kwargs={"lr": lr},
        earlystop_kwargs=None,
    )
    trainer = Trainer(
        model=model,
        settings=settings,
        loss_fn=nn.MSELoss(),
        optimizer=torch.optim.Adam,
        traindataloader=trainloader,
        validdataloader=validloader,
        reporters=[track],
        progress=False,
    )
    trainer.loop()
    return train_curve, val_curve, val_loss


def early_stop_epochs(
    val_loss: list[float], patience: int = PATIENCE
) -> tuple[int, int]:
    """Where early stopping would pick and stop on a recorded val-loss curve: the epoch
    with the lowest loss so far, and the epoch `patience` epochs without improvement
    later (or the last epoch). The same rule as the Trainer's EarlyStopping."""
    best, best_epoch, waited = float("inf"), 0, 0
    for epoch, loss in enumerate(val_loss, start=1):
        if loss < best:
            best, best_epoch, waited = loss, epoch, 0
        else:
            waited += 1
            if waited >= patience:
                return best_epoch, epoch
    return best_epoch, len(val_loss)

## 5.2 When to stop: the curve, and early stopping

Train too briefly and a model is still learning; train too long and it starts fitting
the train windows at the cost of the val windows. You met the standard answer in lesson 1
(`1_representation_and_head/03_training`): **early stopping**. After every epoch the
Trainer computes the val loss; once it has not dropped for `PATIENCE` epochs in a row,
training ends and the weights of the epoch with the lowest val loss are kept.
`MAX_EPOCHS` is then only a ceiling, not a choice.

To see what that rule does, the cell below trains `gate` and its capacity-matched
control `deeper` (seed 0) for a fixed 90 epochs with early stopping off, so the whole
curve is visible, and marks where early stopping with `PATIENCE = 10` would pick its
epoch (dot) and stop (dashed line). Early stopping watches the val *loss*, the MSE the
model trains on, not the val MAE plotted here; the two mostly move together, not
always.

Before you run it: where do you expect the val curves to flatten, and do you expect
train to pull away from val?

In [ ]:
CURVE_EPOCHS = param(90, test=8)

fig, ax = plt.subplots(figsize=(9, 3.2))
colors = {"deeper": "C0", "gate": "C1"}
for name in ("deeper", "gate"):
    train_curve, val_curve, val_loss = mae_curve(name, epochs=CURVE_EPOCHS)
    best_epoch, stop_epoch = early_stop_epochs(val_loss)
    epochs_axis = range(1, len(val_curve) + 1)
    ax.plot(epochs_axis, train_curve, color=colors[name], lw=0.8, ls=":", alpha=0.6)
    ax.plot(epochs_axis, val_curve, color=colors[name], lw=1.4, label=name)
    ax.axvline(stop_epoch, color=colors[name], lw=1, ls="--")
    ax.plot(best_epoch, val_curve[best_epoch - 1], "o", color=colors[name])
    print(
        f"{name:7s} best epoch {best_epoch:3d} (val MAE {val_curve[best_epoch - 1]:.4f})"
        f"   early stopping stops at epoch {stop_epoch:3d}"
        f"   val MAE at epoch {len(val_curve)}: {val_curve[-1]:.4f}"
    )
ax.set_title(
    f"Seed 0: dot = best val loss, dashed = where early stopping "
    f"(patience {PATIENCE}) stops"
)
ax.set_xlabel("epoch")
ax.set_ylabel("MAE  (dotted = train, solid = val)")
ax.legend(fontsize=8)
plt.tight_layout()

Both val curves drop steeply over the first 20 epochs, bouncing as they go, then flatten
into a noisy plateau between about 0.15 and 0.17 that lasts to epoch 90. Train (dotted)
sits about 0.03 below val, and that gap does not widen over the plateau: a stable
generalisation gap, not runaway overfitting.

Early stopping picks a low point on that plateau: epoch 35 for `deeper` (val MAE 0.1552)
and epoch 49 for `gate` (0.1488), and stops 10 epochs later, at 45 and 59. Training on
to epoch 90 buys nothing: `deeper` ends at the same 0.1552, `gate` at 0.1597, worse
than its chosen epoch. The dot is the lowest val *loss* up to the stop, so on this noisy
plateau the MAE curve can still dip below it once by chance (`deeper` near epoch 59);
that bouncing is also why `PATIENCE` is 10 and not 2.

Every run below uses this rule: the same `PATIENCE`, the same ceiling `MAX_EPOCHS`, so
each model trains as long as its own val loss keeps improving, and no model gets an
epoch count picked by hand. For seed 0, `run()` picks the same best epochs as the dots
above (35 for `deeper`, 49 for `gate`): the curve and the runs are the same training.

One caveat holds for every row: the stop epoch is chosen on val and the MAE is reported
on val, so each number is slightly optimistic, equally for every model; this notebook
has no separate test split to report on instead.

## 5.3 The full family, trained with early stopping

In [ ]:
# (label, mechanism, make_model/run kwargs) -- sparse_moe appears twice: once at the
# same N as dense moe (does routing beat blending at matched capacity?), once at 16x
# the experts (does the extra capacity a router can afford almost for free help?).
MODEL_CONFIGS = [
    ("plain", "plain", {}),
    ("deeper", "deeper", {}),
    ("gate", "gate", {}),
    ("highway", "highway", {}),
    ("moe", "moe", {"n_experts": 4}),
    ("sparse_moe (N=4)", "sparse_moe", {"n_experts": 4, "k": 1}),
    ("sparse_moe (N=64)", "sparse_moe", {"n_experts": 64, "k": 1}),
]

results: dict[str, dict] = {}
runs: dict[str, list[Run]] = {}
for label, mechanism, kwargs in MODEL_CONFIGS:
    runs[label] = [run(mechanism, seed, MAX_EPOCHS, **kwargs) for seed in SEEDS]
    scores = [r.mae for r in runs[label]]
    params = sum(p.numel() for p in make_model(mechanism, **kwargs).parameters())
    results[label] = {
        "mae": float(np.mean(scores)),
        "std": float(np.std(scores)),
        "best_epoch": float(np.mean([r.best_epoch for r in runs[label]])),
        "stopped_epoch": float(np.mean([r.stopped_epoch for r in runs[label]])),
        "params": params,
    }
    print(
        f"{label:20s} {results[label]['mae']:.5f} +/- {results[label]['std']:.5f}   "
        f"best epochs {[r.best_epoch for r in runs[label]]}   {params:>8,} params"
    )

## 5.4 The table

In [ ]:
order = sorted(results, key=lambda k: results[k]["mae"])
print(
    f"{'model':20s} {'val MAE':>9} {'+/-':>7} {'best ep':>8} {'stop ep':>8} {'params':>9}"
)
print("-" * 66)
for name in order:
    r = results[name]
    print(
        f"{name:20s} {r['mae']:9.5f} {r['std']:7.5f} {r['best_epoch']:8.1f} "
        f"{r['stopped_epoch']:8.1f} {r['params']:9,}"
    )

Read every gap against the seed spread: the std over 5 seeds runs from 0.003 to 0.012.

- **`gate` against `deeper`, same 19,632 params:** 0.1593 ± 0.0116 against
  0.1605 ± 0.0109. The gap (0.0012) is a tenth of either std: no sign that the gate does
  anything a second plain layer does not.
- **`highway` and dense `moe` have the lowest MAE** (0.1555 ± 0.0034 and
  0.1572 ± 0.0030), at 1.8x and 2.1x `plain`'s parameters. Against `plain`
  (0.1637 ± 0.0038) that is a gap of about twice the std, the clearest gain in the table;
  against `deeper` and `gate` it sits inside their spread.
- **Routing loses to blending at matched capacity.** `sparse_moe (N=4)` has exactly dense
  `moe`'s 32,372 params and scores 0.1655 ± 0.0048 against 0.1572 ± 0.0030, a gap of
  about twice the std, the wrong way for routing.
- **Sixteen times the experts made it worse.** `sparse_moe (N=64)`, at 285,872 params
  (18.5x `plain`, 8.8x dense `moe`), is the worst row at 0.1782 ± 0.0110. It also stops
  earliest: its best epoch is 23 on average, against 32 to 38 for every other model.

The best and stop columns show that early stopping ended every run long before the
ceiling of 200 epochs, and the stop epoch is always the best epoch plus `PATIENCE`.

A series with real daily and weekly structure does not make the gate pay for itself
here, and routing does worse than blending. Routing is a bet that the structure exists
*and that this architecture can find and use it*; the next cell checks the second half
of that bet for the 64-expert router.

### Where do the 64 experts' windows go?

A top-1 router sends each window to one expert. If nearly all windows go to a few
experts, the rest are parameters that never see data. Two mechanisms could produce that:
the router *learns* to favour a few experts during training (router collapse, a known
failure of top-1 routing), or the router never learns at all, and its choices are
whatever its random initial weights give. They predict different things: collapse means
spread-out usage before training, concentrated usage after it, and router weights that
moved.

The cell below takes each seed's trained `sparse_moe (N=64)` from §5.3, counts how many
of the 361 val windows its router sends to each expert, does the same for the untrained
model with the same seed, and measures how far the router's weights moved in training.
Predict first: which of the two mechanisms do you expect?

In [ ]:
N_ROUTED = 64


def expert_counts(model: nn.Module) -> torch.Tensor:
    """How many val windows the router of a trained Linear -> GELU -> SparseMoE -> Linear
    model sends to each expert. SparseMoE routes each row to the argmax of
    `router(h)` (top-1), where h is the hidden vector after the first Linear -> GELU."""
    inp, act, moe = model[0], model[1], model[2]
    with torch.no_grad():
        chosen = moe.router(act(inp(xva))).argmax(dim=-1)
    return torch.bincount(chosen, minlength=N_ROUTED)


def describe(counts: torch.Tensor) -> str:
    top = counts.sort(descending=True).values
    return (
        f"{int((counts > 0).sum()):2d} of {N_ROUTED} experts used   "
        f"top expert {int(top[0]):3d} of {len(xva)} windows   "
        f"top five {int(top[:5].sum()):3d}"
    )


for seed, r in zip(SEEDS, runs["sparse_moe (N=64)"]):
    trained = r.model
    torch.manual_seed(seed)
    untrained = make_model("sparse_moe", n_experts=N_ROUTED, k=1)
    router_change = (trained[2].router.weight - untrained[2].router.weight).abs().max()
    print(f"seed {seed}  untrained: {describe(expert_counts(untrained))}")
    print(f"        trained:   {describe(expert_counts(trained))}")
    print(
        f"        largest change in a router weight during training: {router_change:.2e}"
    )

**The router weights did not move at all**: the largest change is exactly 0 for every
seed. The reason is in `SparseMoE.forward`: with `k=1`, the weight of the chosen expert
is a softmax over one logit, which is always exactly 1, so the loss has no gradient with
respect to the router. The router stays the random projection it started as; what
training changes is the first Linear → GELU, which moves the hidden vectors the router
looks at.

**Usage is concentrated before any training.** The untrained models send the val
windows to 9 to 16 of the 64 experts, and their five most-used experts take 316 to 343
of the 361 windows. After training it is 5 to 18 experts. Seed 0 concentrated further (5
experts; the top one takes 295 windows, the top five all 361); the other four seeds
spread slightly wider than at the start (15 to 18 experts).

So this is not router collapse learned during training: at `k=1` this router cannot
learn, and most of its 64 experts receive no val window from the start. That fits the
table, where the 64-expert model holds 18.5x `plain`'s parameters but uses at most 18
experts on val; it does not show that unused experts are the whole reason for its higher
MAE (E6 tests that).

## 5.5 Is sparse routing cheap to scale?

§5.4 answered "does it forecast better": no, not at `k=1` on this task, and at
`k=1` this router never learns where to send a window. This section asks a separate, still real question: even given that verdict,
does the mechanism deliver on its actual production promise -- holding far more
capacity than any single input pays for in compute? `MoE` is *dense*: every expert
runs on every input, so adding experts always costs more compute -- by construction,
it cannot show that decoupling. `vectormesh`'s `SparseMoE` routes each example
to its top-`k` experts with real gather/scatter -- an expert with zero examples routed
to it this batch does zero work, not masked-out work -- so compute per example scales
with `k`, not with the number of experts `N`.

The test: fix `k=1` (Switch-style) and a fixed batch of this notebook's own windows,
then grow `N` from 4 to 256 experts. If the sparse claim is real, wall-clock time per
forward pass should grow far less than the parameter count, which grows 64x. Dense `MoE`,
run on the identical batch and the identical `N` values, is the control here -- it has
no reason to stay flat, since it always runs every expert regardless of how many
exist.

In [ ]:
import time

from vectormesh.components import SparseMoE

N_EXPERTS_TO_CHECK = param((4, 16, 64, 256), test=(2, 4))
ROUTING_BATCH = 64
REPS = param(30, test=3)

# a real batch of this notebook's own windows, projected to HIDDEN once -- the
# component under test is the mixture-of-experts block, not the input projection
probe_input = nn.Linear(W, HIDDEN)
probe_x = probe_input(xtr[:ROUTING_BATCH])


def time_forward(model: nn.Module, x: torch.Tensor, reps: int) -> float:
    for _ in range(5):
        model(x)
    start = time.perf_counter()
    for _ in range(reps):
        model(x)
    return (time.perf_counter() - start) / reps * 1000


sparse_timings: dict[int, dict] = {}
dense_timings: dict[int, dict] = {}
for n_experts in N_EXPERTS_TO_CHECK:
    torch.manual_seed(0)
    sparse = SparseMoE(
        experts=[nn.Linear(HIDDEN, HIDDEN) for _ in range(n_experts)],
        hidden_size=HIDDEN,
        out_size=HIDDEN,
        k=1,
    )
    torch.manual_seed(0)
    dense = MoE(
        experts=[nn.Linear(HIDDEN, HIDDEN) for _ in range(n_experts)],
        hidden_size=HIDDEN,
        out_size=HIDDEN,
    )

    sparse_ms = time_forward(sparse, probe_x, REPS)
    dense_ms = time_forward(dense, probe_x, REPS)
    params = sum(p.numel() for p in sparse.parameters())  # identical count, both sides

    sparse_timings[n_experts] = {"ms": sparse_ms, "params": params}
    dense_timings[n_experts] = {"ms": dense_ms, "params": params}
    print(
        f"N={n_experts:4d} experts   sparse {sparse_ms:7.3f} ms   "
        f"dense {dense_ms:7.3f} ms   {params:>9,} params"
    )

Measured on this machine (batch of 64 of this notebook's own windows, hidden=64, k=1,
30 reps after 5 warmup calls); the numbers vary a little from run to run:

| N experts | sparse ms/fwd | dense ms/fwd | params |
|---|---|---|---|
| 4 | 0.865 | 0.089 | 16,900 |
| 16 | 1.243 | 0.184 | 67,600 |
| 64 | 1.484 | 0.479 | 270,400 |
| 256 | 2.935 | 1.723 | 1,081,600 |

From N=4 to N=256 the parameter count grows **64x**. Dense `MoE` grows **19x**
(0.089 → 1.723 ms), close to proportional with `N`, as it must: it runs every expert on
every example. `SparseMoE` grows **3.4x** (0.865 → 2.935 ms): each example still passes
through exactly one expert, and the time that is added goes to routing (one `argsort`
and a loop over the experts that received at least one example).

But the sparse version is **slower than dense at every N measured here**: 9.7x slower at
N=4, 1.7x slower at N=256. On a CPU, with experts this small and a batch of 64, the
per-expert gather and scatter cost more than running every small expert densely. The
growth rates say the two lines would cross somewhere above N=256; this table does not
show where.

So the two questions get two different answers. "Does it forecast better here" (§5.4)
is no. "Does compute decouple from capacity" (this section) is yes in how the time
grows, 3.4x for 64x the parameters; whether that makes sparse routing faster than dense
depends on the expert size, the batch and the hardware, and at this size on this machine
it does not.

`04_search` is where you stop checking one hand-picked configuration at a time and let
a search cover the space properly.

## What to take from this

- Gate, Highway and MoE are one idea -- a learned keep-or-replace decision -- at
  increasing resolution, and each one traces to a specific paper rather than floating
  free in a library's component list.
- The mechanism is general-purpose. It showed up here because this lesson had a window
  and a baseline ready, not because time series needs it more than any other modality.
- **Look at the curve, then let early stopping pick the epoch.** §5.2 shows the
  train/val curve and where early stopping stops on it; §5.3 trains every model with
  that same rule, so no model gets a hand-picked epoch count, and the table reports
  when each one stopped.
- Whether gating (or routing) earns its extra parameters *on this window* is the
  number in the table above, not an opinion -- and "capacity-matched control" is doing
  the real work in that sentence, not "gate", "highway", or "sparse_moe" by name.
- **"Does it forecast better" and "is it cheap to scale" are different questions, and
  this notebook answers both for MoE.** §5.3 trains dense and sparse routing side by
  side and reports the accuracy verdict directly. §5.5 measures the separate claim,
  that sparse compute grows far slower than capacity, and finds the growth holds while
  sparse stays slower than dense at every size measured on this CPU.
- **A negative result can still have a checkable cause.** §5.4 did not stop at "16x
  more experts did not help": it counted where the router sends the val windows and
  whether its weights moved, and found that at `k=1` the router never learns and uses
  5 to 18 of its 64 experts. "It didn't work" and "it didn't work, and here is why" are
  different levels of finished.

## Exercises

**E1 -- predict, then measure.** Before re-running this notebook, settle on a ranking
you expect for all seven trained models, and say whether you expect this series'
structure to change the §5.4 verdict that `gate` does no better than `deeper`. Check both.

**E2 -- turn the gate off.** Force `Gate`'s output to equal its input (patch the
forward pass, or initialise its linear layer to a large positive bias so the sigmoid
saturates near 1) and confirm the block's score converges toward `deeper`'s -- the
collapse argument from `1_representation_and_head/02_nonlinearity` §2.1, for a gate instead of a
stacked linear layer.

**E3 -- more experts, dense vs sparse.** Re-run `make_model("moe")` with 2, 4, 8 and 16
experts (dense), each trained with the same early stopping as §5.3, and compare against
`sparse_moe (N=64)`'s row already in §5.4's table. Does the dense router find anything
to split on as N grows, or does the score just track the parameter count -- and does
that answer differ from the routed version?

**E4 -- a different window.** Swap in `pedestrian_counts` (Monash `monash_tsf`, config
`pedestrian_counts`; `australian_electricity.py` shows how to load a config and cut it in
time) and re-run the whole notebook. Does the ranking change on a series with an even sharper daily cycle?

**E5 -- k>1.** Re-run §5.5's timing sweep with `k=2` and `k=4` instead of `k=1`.
Predict first: does sparse time grow faster with `N` than at `k=1`, and does it get
closer to dense or further from it? Then re-run §5.4's
`sparse_moe (N=64)` row at `k=2` and `k=4` -- does blending more routed experts change
the accuracy verdict, not just the timing one? With `k>1` the top-`k` softmax has more
than one logit, so rerun §5.4's router cell too: do the router weights move now?

**E6 -- give the router a gradient.** §5.4 found that at `k=1` `SparseMoE`'s router
never changes, because the chosen expert's weight is always exactly 1. The Switch
Transformer scales the chosen expert's output by its probability under the softmax over
*all* N router logits, which does depend on the router weights; it also adds an
auxiliary load-balancing loss that pushes the router to spread windows over the experts.
Subclass `SparseMoE` to do the first, add the second to the loss, and re-check with
§5.4's router cell and the table: does a router that learns, and spreads its load,
change the accuracy verdict, or only the usage?
